# MIT-Adobe FiveK Relative Depth Evaluation

This notebook evaluates the effect of five input pipelines on one fixed
DepthAnythingV2 relative-depth model:

- Dark baseline
- Gamma correction
- CLAHE
- Multi-Scale Retinex (MSR)
- LLFormer using the MIT-Adobe FiveK checkpoint

For each paired image, the relative depth predicted from the high-quality image
is used as a pseudo-reference. Every depth map is independently normalised to
[0, 1] before MAE, RMSE, and AbsRel are calculated.

## 1. Environment Setup

In [1]:
import os
import sys
import random
import shutil
import subprocess
from collections import OrderedDict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import torchvision.transforms.functional as TF

from PIL import Image
from skimage import img_as_ubyte
from tqdm import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

WORK_DIR = Path("/kaggle/working")
DEPTH_REPO = WORK_DIR / "Depth-Anything-V2"
LLFORMER_REPO = WORK_DIR / "LLFormer"


def run_command(command, cwd=None):
    """Run a shell command and display its output."""
    print(f"\nRunning: {command}")

    result = subprocess.run(
        command,
        cwd=str(cwd) if cwd else None,
        shell=True,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT
    )

    print(result.stdout)

    if result.returncode != 0:
        raise RuntimeError(
            f"Command failed with exit code "
            f"{result.returncode}:\n{command}"
        )


if not DEPTH_REPO.is_dir():
    run_command(
        "git clone "
        "https://github.com/DepthAnything/"
        "Depth-Anything-V2.git",
        cwd=WORK_DIR
    )
else:
    print(
        "Depth Anything V2 already exists: "
        f"{DEPTH_REPO}"
    )


if not LLFORMER_REPO.is_dir():
    run_command(
        "git clone "
        "https://github.com/TaoWangzj/LLFormer.git",
        cwd=WORK_DIR
    )
else:
    print(
        f"LLFormer already exists: {LLFORMER_REPO}"
    )


run_command(
    f'"{sys.executable}" -m pip install '
    f'opencv-python-headless '
    f'natsort yacs gdown scikit-image -q'
)

warmup_dir = (
    LLFORMER_REPO
    / "pytorch-gradual-warmup-lr"
)

if warmup_dir.is_dir():
    run_command(
        f'"{sys.executable}" setup.py install',
        cwd=warmup_dir
    )


for repo_path in [
    str(DEPTH_REPO),
    str(LLFORMER_REPO)
]:
    if repo_path not in sys.path:
        sys.path.insert(0, repo_path)


DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print(f"\nUsing device: {DEVICE}")
print("Environment setup completed.")


Running: git clone https://github.com/DepthAnything/Depth-Anything-V2.git
Cloning into 'Depth-Anything-V2'...


Running: git clone https://github.com/TaoWangzj/LLFormer.git
Cloning into 'LLFormer'...


Running: "/usr/bin/python3" -m pip install opencv-python-headless natsort yacs gdown scikit-image -q


Running: "/usr/bin/python3" setup.py install
running install
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:90: SetuptoolsDeprecationWarning: setup.py install is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` directly.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        This deprecation is overdue, please update your project and remove deprecated
        calls to avoid build errors in the future.

        See https://blog.ganssle.io/articles/2021/10/setup-py-deprecated.html for details.
        ***************************

## 2. Download LLFormer MIT-Adobe FiveK Weights

In [2]:
LLFORMER_CHECKPOINT_DIR = (
    LLFORMER_REPO
    / "checkpoints"
    / "MIT"
)

LLFORMER_CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


def find_llformer_checkpoint(root_dir):
    candidates = list(
        root_dir.rglob("model_bestPSNR.pth")
    )

    if not candidates:
        return None

    candidates.sort(
        key=lambda path: (
            "models" not in path.parts,
            len(path.parts)
        )
    )

    return candidates[0]


weights_path = find_llformer_checkpoint(
    LLFORMER_CHECKPOINT_DIR
)

if weights_path is None:
    command = (
        f'"{sys.executable}" -m gdown --folder '
        f'https://drive.google.com/drive/folders/'
        f'1CmZC2drX2t3H9U4zq4DvOlsP03J7AYXy '
        f'-O "{LLFORMER_CHECKPOINT_DIR}"'
    )

    run_command(command)

    weights_path = find_llformer_checkpoint(
        LLFORMER_CHECKPOINT_DIR
    )


if weights_path is None:
    print("\nDownloaded files:")

    for path in LLFORMER_CHECKPOINT_DIR.rglob("*"):
        if path.is_file():
            print(path)

    raise FileNotFoundError(
        "model_bestPSNR.pth could not be found "
        "under the MIT checkpoint directory."
    )


print("LLFormer checkpoint ready:")
print(weights_path)


Running: "/usr/bin/python3" -m gdown --folder https://drive.google.com/drive/folders/1CmZC2drX2t3H9U4zq4DvOlsP03J7AYXy -O "/kaggle/working/LLFormer/checkpoints/MIT"
Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1tbxMUTlWNSwnoud4374UZwRgFXRMFmsc
From (redirected): https://drive.google.com/uc?id=1tbxMUTlWNSwnoud4374UZwRgFXRMFmsc&confirm=t&uuid=8be05e60-ed9b-44be-a193-ca96305f3890
To: /kaggle/working/LLFormer/checkpoints/MIT/models/model_bestPSNR.pth
Retrieving folder 1lqs14CabYFK9iY5Tj_UWCkL5HyXJsyT4 models
Processing file 1tbxMUTlWNSwnoud4374UZwRgFXRMFmsc model_bestPSNR.pth
Processing file 1pVrZms4mHhsAN6Bir5kbiqQwKvHXNgfQ model_bestSSIM.pth

  0%|          | 0.00/296M [00:00<?, ?B/s]
  0%|          | 1.05M/296M [00:00<00:30, 9.52MB/s]
  2%|▏         | 4.72M/296M [00:00<00:31, 9.30MB/s]
  4%|▍         | 13.1M/296M [00:00<00:17, 16.1MB/s]
 

## 3. Dataset Paths and Sample Definition

In [3]:
HIGH_DIR = Path(
    "/kaggle/input/datasets/mavislei/"
    "quant1-set/high"
)

LOW_DIR = Path(
    "/kaggle/input/datasets/mavislei/"
    "quant1-set/low"
)

DEPTH_MODEL_PATH = Path(
    "/kaggle/input/models/artemmmtry/"
    "depth-anything-v2/pytorch/"
    "small-model/1/"
    "depth_anything_v2_vits.pth"
)

OUTPUT_ROOT = (
    WORK_DIR
    / "mit_fivek_relative_depth_results"
)

LLFORMER_OUTPUT_DIR = (
    WORK_DIR
    / "mit_fivek_llformer_enhanced"
)

CSV_DIR = OUTPUT_ROOT / "csv"


for required_path in [
    HIGH_DIR,
    LOW_DIR,
    DEPTH_MODEL_PATH
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Required path not found: "
            f"{required_path}"
        )


SUPPORTED_EXTENSIONS = {
    ".png",
    ".jpg",
    ".jpeg",
    ".bmp",
    ".tif",
    ".tiff"
}

high_files = {
    path.name: path
    for path in HIGH_DIR.iterdir()
    if (
        path.is_file()
        and path.suffix.lower()
        in SUPPORTED_EXTENSIONS
    )
}

low_files = {
    path.name: path
    for path in LOW_DIR.iterdir()
    if (
        path.is_file()
        and path.suffix.lower()
        in SUPPORTED_EXTENSIONS
    )
}

paired_names = sorted(
    set(high_files)
    & set(low_files)
)

if not paired_names:
    raise RuntimeError(
        "No paired filenames were found "
        "between the high and low folders."
    )


SAMPLE_SIZE = min(
    100,
    len(paired_names)
)

sampled_images = sorted(
    random.Random(SEED).sample(
        paired_names,
        SAMPLE_SIZE
    )
)


shutil.rmtree(
    OUTPUT_ROOT,
    ignore_errors=True
)

shutil.rmtree(
    LLFORMER_OUTPUT_DIR,
    ignore_errors=True
)

OUTPUT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

LLFORMER_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

CSV_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print(
    f"Available paired images: "
    f"{len(paired_names)}"
)

print(
    f"Evaluation sample: "
    f"{len(sampled_images)}"
)

print(
    "First five sampled images:",
    sampled_images[:5]
)

Available paired images: 500
Evaluation sample: 100
First five sampled images: ['a4504.png', 'a4513.png', 'a4514.png', 'a4516.png', 'a4517.png']


## 4. Load Models and Define Processing Functions

In [4]:
from depth_anything_v2.dpt import DepthAnythingV2


depth_model = DepthAnythingV2(
    encoder="vits",
    features=64,
    out_channels=[
        48,
        96,
        192,
        384
    ]
)

depth_checkpoint = torch.load(
    str(DEPTH_MODEL_PATH),
    map_location=DEVICE
)

depth_model.load_state_dict(
    depth_checkpoint
)

depth_model = (
    depth_model
    .to(DEVICE)
    .eval()
)

print(
    "DepthAnythingV2 relative-depth "
    f"model loaded on {DEVICE}"
)


def normalise_relative_depth(depth):
    """Normalise one relative-depth map to [0, 1]."""
    depth = np.asarray(
        depth,
        dtype=np.float32
    )

    valid = np.isfinite(depth)

    output = np.zeros_like(
        depth,
        dtype=np.float32
    )

    if valid.sum() == 0:
        return output

    depth_min = depth[valid].min()
    depth_max = depth[valid].max()

    if depth_max - depth_min < 1e-8:
        return output

    output[valid] = (
        depth[valid] - depth_min
    ) / (
        depth_max - depth_min
    )

    return output


def run_relative_depth(
    image_bgr,
    input_size=518
):
    """Run relative-depth inference."""
    if image_bgr is None:
        raise ValueError(
            "Input image is None."
        )

    image_rgb = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2RGB
    )

    with torch.no_grad():
        depth = depth_model.infer_image(
            image_rgb,
            input_size=input_size
        )

    return normalise_relative_depth(
        depth
    )


def depth_to_vis(depth):
    """Convert relative depth to a TURBO image."""
    depth_norm = normalise_relative_depth(
        depth
    )

    depth_u8 = np.clip(
        depth_norm * 255.0,
        0,
        255
    ).astype(np.uint8)

    return cv2.applyColorMap(
        depth_u8,
        cv2.COLORMAP_TURBO
    )


def apply_gamma(
    image_bgr,
    gamma=2.2
):
    inv_gamma = 1.0 / gamma

    table = np.array([
        (
            (value / 255.0)
            ** inv_gamma
        ) * 255.0
        for value in range(256)
    ]).astype(np.uint8)

    return cv2.LUT(
        image_bgr,
        table
    )


def apply_clahe(image_bgr):
    lab = cv2.cvtColor(
        image_bgr,
        cv2.COLOR_BGR2LAB
    )

    l_channel, a_channel, b_channel = (
        cv2.split(lab)
    )

    clahe = cv2.createCLAHE(
        clipLimit=2.0,
        tileGridSize=(8, 8)
    )

    l_enhanced = clahe.apply(
        l_channel
    )

    enhanced_lab = cv2.merge([
        l_enhanced,
        a_channel,
        b_channel
    ])

    return cv2.cvtColor(
        enhanced_lab,
        cv2.COLOR_LAB2BGR
    )


def apply_msr(
    image_bgr,
    sigmas=(15, 80, 250)
):
    """
    Multi-Scale Retinex with
    per-channel Min-Max normalisation.
    """
    image = (
        image_bgr
        .astype(np.float32)
        + 1.0
    )

    msr = np.zeros_like(
        image,
        dtype=np.float32
    )

    for sigma in sigmas:
        blurred = cv2.GaussianBlur(
            image,
            (0, 0),
            sigma
        )

        msr += (
            np.log(image)
            - np.log(blurred + 1e-6)
        )

    msr /= len(sigmas)

    output = np.zeros_like(
        msr,
        dtype=np.float32
    )

    for channel_idx in range(3):
        channel = (
            msr[:, :, channel_idx]
        )

        channel_min = channel.min()
        channel_max = channel.max()

        if (
            channel_max
            - channel_min
            < 1e-8
        ):
            output[
                :,
                :,
                channel_idx
            ] = 0

            continue

        output[
            :,
            :,
            channel_idx
        ] = (
            (
                channel
                - channel_min
            )
            / (
                channel_max
                - channel_min
            )
            * 255.0
        )

    return np.clip(
        output,
        0,
        255
    ).astype(np.uint8)


def load_llformer():
    """Load LLFormer with the MIT checkpoint."""
    from model.LLFormer import (
        LLFormer as LLFormerModel
    )

    model = LLFormerModel(
        inp_channels=3,
        out_channels=3,
        dim=16,
        num_blocks=[
            2,
            4,
            8,
            16
        ],
        num_refinement_blocks=2,
        heads=[
            1,
            2,
            4,
            8
        ],
        ffn_expansion_factor=2.66,
        bias=False,
        LayerNorm_type="WithBias",
        attention=True,
        skip=False
    )

    checkpoint = torch.load(
        str(weights_path),
        map_location="cpu"
    )

    state_dict = checkpoint.get(
        "state_dict",
        checkpoint
    )

    try:
        model.load_state_dict(
            state_dict
        )

    except RuntimeError:
        cleaned_state_dict = (
            OrderedDict()
        )

        for key, value in (
            state_dict.items()
        ):
            cleaned_key = (
                key[7:]
                if key.startswith("module.")
                else key
            )

            cleaned_state_dict[
                cleaned_key
            ] = value

        model.load_state_dict(
            cleaned_state_dict
        )

    return (
        model
        .to(DEVICE)
        .eval()
    )


def apply_llformer(
    image_bgr,
    model,
    max_size=512
):
    """
    Resize the longest side to at most 512,
    pad to a multiple of 16, infer, crop,
    and restore the original resolution.
    """
    if image_bgr is None:
        raise ValueError(
            "Input image is None."
        )

    original_height, original_width = (
        image_bgr.shape[:2]
    )

    resized = image_bgr
    height, width = resized.shape[:2]

    if max(height, width) > max_size:
        scale = (
            max_size
            / max(height, width)
        )

        resized = cv2.resize(
            resized,
            (
                max(
                    1,
                    int(round(width * scale))
                ),
                max(
                    1,
                    int(round(height * scale))
                )
            ),
            interpolation=cv2.INTER_AREA
        )

    image_rgb = cv2.cvtColor(
        resized,
        cv2.COLOR_BGR2RGB
    )

    input_tensor = TF.to_tensor(
        Image.fromarray(image_rgb)
    ).unsqueeze(0).to(DEVICE)

    height, width = (
        input_tensor.shape[2:]
    )

    multiple = 16

    padded_height = (
        (
            height
            + multiple
            - 1
        )
        // multiple
        * multiple
    )

    padded_width = (
        (
            width
            + multiple
            - 1
        )
        // multiple
        * multiple
    )

    input_tensor = F.pad(
        input_tensor,
        (
            0,
            padded_width - width,
            0,
            padded_height - height
        ),
        mode="reflect"
    )

    with torch.no_grad():
        output = model(
            input_tensor
        )

    output = torch.clamp(
        output,
        0,
        1
    )[
        :,
        :,
        :height,
        :width
    ]

    output = img_as_ubyte(
        output.permute(
            0,
            2,
            3,
            1
        ).cpu().numpy()[0]
    )

    output = cv2.cvtColor(
        output,
        cv2.COLOR_RGB2BGR
    )

    return cv2.resize(
        output,
        (
            original_width,
            original_height
        ),
        interpolation=cv2.INTER_LINEAR
    )


def compute_metrics(
    prediction,
    pseudo_reference
):
    """
    Compare independently normalised
    relative-depth maps.
    """
    prediction = (
        normalise_relative_depth(
            prediction
        )
    )

    pseudo_reference = (
        normalise_relative_depth(
            pseudo_reference
        )
    )

    if (
        prediction.shape
        != pseudo_reference.shape
    ):
        prediction = cv2.resize(
            prediction,
            (
                pseudo_reference.shape[1],
                pseudo_reference.shape[0]
            ),
            interpolation=cv2.INTER_CUBIC
        )

        prediction = (
            normalise_relative_depth(
                prediction
            )
        )

    mask = (
        np.isfinite(
            pseudo_reference
        )
        & np.isfinite(
            prediction
        )
    )

    valid_pixels = int(
        mask.sum()
    )

    if valid_pixels == 0:
        return None

    pred_valid = prediction[mask]
    ref_valid = pseudo_reference[mask]

    mae = float(
        np.mean(
            np.abs(
                pred_valid
                - ref_valid
            )
        )
    )

    rmse = float(
        np.sqrt(
            np.mean(
                (
                    pred_valid
                    - ref_valid
                ) ** 2
            )
        )
    )

    absrel_mask = (
        mask
        & (
            pseudo_reference
            > 1e-3
        )
    )

    if absrel_mask.sum() == 0:
        absrel = np.nan

    else:
        absrel = float(
            np.mean(
                np.abs(
                    prediction[
                        absrel_mask
                    ]
                    - pseudo_reference[
                        absrel_mask
                    ]
                )
                / pseudo_reference[
                    absrel_mask
                ]
            )
        )

    return {
        "mae": mae,
        "rmse": rmse,
        "absrel": absrel,
        "valid_pixels": valid_pixels
    }


PIPELINES = [
    "dark",
    "gamma",
    "clahe",
    "msr",
    "llformer"
]

print(
    "Models and processing "
    "functions are ready."
)

xFormers not available
xFormers not available


DepthAnythingV2 relative-depth model loaded on cuda
Models and processing functions are ready.


## 5. Pre-compute LLFormer Images

In [5]:
def precompute_llformer_images(
    image_names,
    input_dir,
    output_dir
):
    model = load_llformer()

    try:
        for filename in tqdm(
            image_names,
            desc="LLFormer: MIT FiveK sample"
        ):
            output_path = (
                output_dir
                / filename
            )

            if output_path.is_file():
                continue

            input_path = (
                input_dir
                / filename
            )

            image = cv2.imread(
                str(input_path)
            )

            if image is None:
                raise FileNotFoundError(
                    "Unable to read image: "
                    f"{input_path}"
                )

            enhanced = apply_llformer(
                image,
                model,
                max_size=512
            )

            success = cv2.imwrite(
                str(output_path),
                enhanced
            )

            if not success:
                raise IOError(
                    "Unable to save image: "
                    f"{output_path}"
                )

    finally:
        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


precompute_llformer_images(
    sampled_images,
    LOW_DIR,
    LLFORMER_OUTPUT_DIR
)

print(
    "LLFormer pre-computation "
    "completed."
)

LLFormer: MIT FiveK sample: 100%|██████████| 100/100 [01:00<00:00,  1.66it/s]

LLFormer pre-computation completed.


## 6. Relative Depth Evaluation

In [6]:
records = []


for filename in tqdm(
    sampled_images,
    desc="MIT FiveK evaluation"
):
    high_path = (
        HIGH_DIR
        / filename
    )

    low_path = (
        LOW_DIR
        / filename
    )

    high_bgr = cv2.imread(
        str(high_path),
        cv2.IMREAD_COLOR
    )

    low_bgr = cv2.imread(
        str(low_path),
        cv2.IMREAD_COLOR
    )

    if (
        high_bgr is None
        or low_bgr is None
    ):
        print(
            "Skipped unreadable pair: "
            f"{filename}"
        )

        continue

    stem = Path(filename).stem

    image_output_dir = (
        OUTPUT_ROOT
        / stem
    )

    enhanced_dir = (
        image_output_dir
        / "enhanced"
    )

    depth_raw_dir = (
        image_output_dir
        / "depth_raw"
    )

    depth_vis_dir = (
        image_output_dir
        / "depth_vis"
    )

    for directory in [
        enhanced_dir,
        depth_raw_dir,
        depth_vis_dir
    ]:
        directory.mkdir(
            parents=True,
            exist_ok=True
        )

    cv2.imwrite(
        str(
            image_output_dir
            / "high.png"
        ),
        high_bgr
    )

    cv2.imwrite(
        str(
            image_output_dir
            / "dark.png"
        ),
        low_bgr
    )

    pseudo_reference = (
        run_relative_depth(
            high_bgr
        )
    )

    np.save(
        depth_raw_dir
        / "pseudo_reference.npy",
        pseudo_reference
    )

    cv2.imwrite(
        str(
            depth_vis_dir
            / "pseudo_reference.png"
        ),
        depth_to_vis(
            pseudo_reference
        )
    )

    llformer_bgr = cv2.imread(
        str(
            LLFORMER_OUTPUT_DIR
            / filename
        ),
        cv2.IMREAD_COLOR
    )

    if llformer_bgr is None:
        raise FileNotFoundError(
            "Unable to read LLFormer output: "
            f"{LLFORMER_OUTPUT_DIR / filename}"
        )

    pipeline_images = {
        "dark": low_bgr,
        "gamma": apply_gamma(
            low_bgr
        ),
        "clahe": apply_clahe(
            low_bgr
        ),
        "msr": apply_msr(
            low_bgr
        ),
        "llformer": llformer_bgr
    }

    image_records = []

    for pipeline_name in PIPELINES:
        processed_bgr = (
            pipeline_images[
                pipeline_name
            ]
        )

        cv2.imwrite(
            str(
                enhanced_dir
                / f"{pipeline_name}.png"
            ),
            processed_bgr
        )

        predicted_depth = (
            run_relative_depth(
                processed_bgr
            )
        )

        np.save(
            depth_raw_dir
            / f"{pipeline_name}.npy",
            predicted_depth
        )

        cv2.imwrite(
            str(
                depth_vis_dir
                / f"{pipeline_name}.png"
            ),
            depth_to_vis(
                predicted_depth
            )
        )

        metrics = compute_metrics(
            predicted_depth,
            pseudo_reference
        )

        if metrics is None:
            continue

        record = {
            "image": stem,
            "source_filename": filename,
            "pipeline": pipeline_name,
            **metrics
        }

        records.append(record)
        image_records.append(record)

    if image_records:
        pd.DataFrame(
            image_records
        ).to_csv(
            image_output_dir
            / "metrics.csv",
            index=False
        )


results_df = pd.DataFrame(
    records
)

if results_df.empty:
    raise RuntimeError(
        "No valid evaluation results "
        "were produced."
    )


results_df.to_csv(
    CSV_DIR
    / "per_image_metrics.csv",
    index=False
)

print(
    "Completed image pairs: "
    f"{results_df['image'].nunique()}"
)

display(
    results_df.head()
)

MIT FiveK evaluation: 100%|██████████| 100/100 [02:06<00:00,  1.27s/it]

Completed image pairs: 100


,image,source_filename,pipeline,mae,rmse,absrel,valid_pixels
0,a4504,a4504.png,dark,0.039450,0.047549,0.164223,174592
1,a4504,a4504.png,gamma,0.014154,0.020946,0.060321,174592
2,a4504,a4504.png,clahe,0.027587,0.035845,0.114084,174592
3,a4504,a4504.png,msr,0.011747,0.017789,0.048981,174592
4,a4504,a4504.png,llformer,0.016765,0.020086,0.078322,174592


## 7. Save Summary Tables and Package Results

In [7]:
summary_df = (
    results_df
    .groupby(
        "pipeline",
        as_index=False
    )
    .agg(
        images=(
            "image",
            "nunique"
        ),
        mean_mae=(
            "mae",
            "mean"
        ),
        std_mae=(
            "mae",
            "std"
        ),
        mean_rmse=(
            "rmse",
            "mean"
        ),
        std_rmse=(
            "rmse",
            "std"
        ),
        mean_absrel=(
            "absrel",
            "mean"
        ),
        std_absrel=(
            "absrel",
            "std"
        ),
        mean_valid_pixels=(
            "valid_pixels",
            "mean"
        )
    )
)


pipeline_order = {
    pipeline: index
    for index, pipeline
    in enumerate(PIPELINES)
}

summary_df[
    "pipeline_order"
] = summary_df[
    "pipeline"
].map(
    pipeline_order
)

summary_df = (
    summary_df
    .sort_values(
        "pipeline_order"
    )
    .drop(
        columns="pipeline_order"
    )
    .reset_index(
        drop=True
    )
)


dark_mae = (
    results_df[
        results_df["pipeline"]
        == "dark"
    ][
        [
            "image",
            "mae"
        ]
    ]
    .rename(
        columns={
            "mae": "dark_mae"
        }
    )
)


comparison_df = (
    results_df
    .merge(
        dark_mae,
        on="image",
        how="left"
    )
)

comparison_df[
    "better_than_dark_mae"
] = (
    comparison_df["mae"]
    < comparison_df["dark_mae"]
)


win_summary_df = (
    comparison_df[
        comparison_df["pipeline"]
        != "dark"
    ]
    .groupby(
        "pipeline",
        as_index=False
    )
    .agg(
        better_than_dark=(
            "better_than_dark_mae",
            "sum"
        ),
        evaluated_images=(
            "image",
            "nunique"
        )
    )
)

win_summary_df[
    "worse_or_equal_to_dark"
] = (
    win_summary_df[
        "evaluated_images"
    ]
    - win_summary_df[
        "better_than_dark"
    ]
)


summary_df.to_csv(
    CSV_DIR
    / "summary_metrics.csv",
    index=False
)

win_summary_df.to_csv(
    CSV_DIR
    / "mae_comparison_with_dark.csv",
    index=False
)


print(
    "===== MIT-Adobe FiveK "
    "relative-depth results ====="
)

display(summary_df)

print(
    "===== Per-image MAE comparison "
    "with Dark ====="
)

display(win_summary_df)


archive_base = (
    WORK_DIR
    / "mit_fivek_relative_depth_results"
)

archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(OUTPUT_ROOT)
)

print(
    f"Results archive created: "
    f"{archive_path}"
)

===== MIT-Adobe FiveK relative-depth results =====


,pipeline,images,mean_mae,std_mae,mean_rmse,std_rmse,mean_absrel,std_absrel,mean_valid_pixels
0,dark,100,0.014754,0.017111,0.020203,0.018883,0.095397,0.080291,174679.04
1,gamma,100,0.013307,0.014671,0.018968,0.017075,0.087799,0.085137,174679.04
2,clahe,100,0.018801,0.023321,0.025835,0.025810,0.135452,0.270803,174679.04
3,msr,100,0.017822,0.017039,0.025294,0.020370,0.131361,0.218923,174679.04
4,llformer,100,0.011237,0.016835,0.015422,0.018210,0.072282,0.072471,174679.04


===== Per-image MAE comparison with Dark =====


,pipeline,better_than_dark,evaluated_images,worse_or_equal_to_dark
0,clahe,35,100,65
1,gamma,54,100,46
2,llformer,65,100,35
3,msr,33,100,67


Results archive created: /kaggle/working/mit_fivek_relative_depth_results.zip
